<a href="https://colab.research.google.com/github/fce-laboratoriotp/tp-labo/blob/main/notebook_prueba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prueba de Commit Grupal - Laboratorio de Métodos Cuantitativos
Este es un archivo de prueba para verificar la conexión con el repositorio de GitHub y asegurar que los commits de todas las integrantes queden registrados correctamente.

buenasss

In [37]:
import pandas as pd


hola soy miguelina

hola soy aldi

Hola, soy Kay


hola soy mica

# **Análisis Exploratorio y Detección de Variables**

En esta etapa inicial examinamos la base de datos oficial a nivel provincial (`snic-provincias.csv`) para evaluar su estructura, identificar qué información aporta y verificar los registros del fenómeno en cada jurisdicción.

In [38]:
import pandas as pd

# carga de la base de datos a nivel provincial
df_prov = pd.read_csv("snic-provincias.csv", sep=";", encoding="latin1", on_bad_lines="skip")

# miro el tamaño de la tabla
print("dimensiones de la tabla provincial:", df_prov.shape)

# reviso las primeras filas
display(df_prov.head(3))

# reviso los tipos de datos de cada columna
print("\ntipos de datos:")
print(df_prov.dtypes)

# chequeo si hay valores nulos o celdas vacias
print("\nvalores nulos:")
print(df_prov.isnull().sum())

dimensiones de la tabla provincial: (26528, 14)


,provincia_id,provincia_nombre,anio,codigo_delito_snic_id,codigo_delito_snic_nombre,cantidad_hechos,cantidad_victimas,cantidad_victimas_masc,cantidad_victimas_fem,cantidad_victimas_sd,tasa_hechos,tasa_victimas,tasa_victimas_masc,tasa_victimas_fem
0,2,Ciudad AutÃ³noma de Buenos Aires,2000,1,Homicidios dolosos,149,158.0,121.0,37.0,0.0,"4,9750013","5,2755051","8,8257008","2,2783601"
1,2,Ciudad AutÃ³noma de Buenos Aires,2000,2,Homicidios dolosos en grado de tentativa,54,58.0,40.0,18.0,0.0,"1,8030207","1,9365778","2,9175868","1,1083914"
2,2,Ciudad AutÃ³noma de Buenos Aires,2000,3,Muertes en accidentes viales,121,126.0,97.0,29.0,0.0,"4,040102","4,2070484","7,0751481","1,7857418"



tipos de datos:
provincia_id                   int64
provincia_nombre              object
anio                           int64
codigo_delito_snic_id         object
codigo_delito_snic_nombre     object
cantidad_hechos                int64
cantidad_victimas            float64
cantidad_victimas_masc       float64
cantidad_victimas_fem        float64
cantidad_victimas_sd         float64
tasa_hechos                   object
tasa_victimas                 object
tasa_victimas_masc            object
tasa_victimas_fem             object
dtype: object

valores nulos:
provincia_id                     0
provincia_nombre                 0
anio                             0
codigo_delito_snic_id            0
codigo_delito_snic_nombre        0
cantidad_hechos                  0
cantidad_victimas            16688
cantidad_victimas_masc       16688
cantidad_victimas_fem        16688
cantidad_victimas_sd         16688
tasa_hechos                      0
tasa_victimas                16688
tasa_victimas_m

In [39]:
# filtramos los nulos para ver a que delitos corresponden
nulos_df = df_prov[df_prov["cantidad_victimas"].isnull()]

# contamos cuantos de esos nulos pertenecen a la categoria de suicidios
suicidios_nulos = nulos_df[nulos_df["codigo_delito_snic_nombre"].astype(str).str.contains("Suicidios", case=False, na=False)]

print(f"\ntotal de filas con nulos en todo el dataset: {len(nulos_df)}")
print(f"de esos nulos, cuantos corresponden a suicidios: {len(suicidios_nulos)}")

if len(suicidios_nulos) == 0:
    print("ninguno de los nulos afecta a los suicidios. estan en otros delitos.")
else:
    print("hay registros nulos en suicidios. vamos a ver en que anios ocurren:")
    display(suicidios_nulos[["anio", "codigo_delito_snic_nombre"]])


total de filas con nulos en todo el dataset: 16688
de esos nulos, cuantos corresponden a suicidios: 0
ninguno de los nulos afecta a los suicidios. estan en otros delitos.


### **Extracción y acotamiento del universo de estudio**

El siguiente paso metodológico consiste en extraer exclusivamente los registros asociados al delito de **Suicidios** de la base provincial y acotar el horizonte temporal al quinquenio **2020-2024**, en consonancia con los objetivos del trabajo.

In [40]:
# seleccionamos los registros de suicidios para el periodo de estudio
df_prov_suicidios = df_prov[
    (df_prov["codigo_delito_snic_nombre"].astype(str).str.contains("Suicidios", case=False, na=False)) &
    (df_prov["anio"].between(2020, 2024))
].copy()

print(f"cantidad de observaciones seleccionadas para el analisis: {len(df_prov_suicidios)}")
display(df_prov_suicidios.head(3))

cantidad de observaciones seleccionadas para el analisis: 120


,provincia_id,provincia_nombre,anio,codigo_delito_snic_id,codigo_delito_snic_nombre,cantidad_hechos,cantidad_victimas,cantidad_victimas_masc,cantidad_victimas_fem,cantidad_victimas_sd,tasa_hechos,tasa_victimas,tasa_victimas_masc,tasa_victimas_fem
766,2,Ciudad AutÃ³noma de Buenos Aires,2020,31,Suicidios (consumados),206,206.0,144.0,62.0,0.0,"7,17360364586217","7,17360364586217","10,7500391928512","4,04670947040974"
826,2,Ciudad AutÃ³noma de Buenos Aires,2021,31,Suicidios (consumados),236,236.0,176.0,60.0,0.0,"8,20424395806102","8,20424395806102","13,1013891194452","3,91340674449563"
886,2,Ciudad AutÃ³noma de Buenos Aires,2022,31,Suicidios (consumados),242,242.0,171.0,71.0,0.0,"8,14167866611043","8,14167866611043","10,6373052160119","5,20218931572893"


### **Incorporación y análisis de microdatos**

Para dar cumplimiento a los objetivos del trabajo práctico y acceder a variables demográficas clave (como tramos de edad, sexo e identidad de género), incorporamos la base de **microdatos** (`SAT-SS-BU_2017-2024.csv`). Procedemos a cargarla, inspeccionar su estructura y acotarla al mismo período de estudio.

In [41]:
# cargamos la base de microdatos
df_micro = pd.read_csv("SAT-SS-BU_2017-2024.csv", sep=",", encoding="latin1", on_bad_lines="skip")

# inspeccionamos dimensiones y columnas
print("dimensiones de la base de microdatos:")
print(f"filas: {df_micro.shape[0]} | columnas: {df_micro.shape[1]}")

print("\ncolumnas disponibles en microdatos:")
print(df_micro.columns.tolist())

# vista preliminar de los registros
display(df_micro.head(3))

# filtramos los microdatos solo para 2020-2024
df_micro_2020_2024 = df_micro[df_micro["anio"].between(2020, 2024)].copy()

print("victimas en microdatos (2020-2024):", len(df_micro_2020_2024))
print("victimas en base provincial (2020-2024):", df_prov_suicidios["cantidad_victimas"].sum())

dimensiones de la base de microdatos:
filas: 30177 | columnas: 27

columnas disponibles en microdatos:
['id_hecho', 'tipo_persona_id', 'federal', 'provincia_id', 'provincia_nombre', 'departamento_id', 'departamento_nombre', 'localidad_id', 'localidad_nombre', 'anio', 'mes', 'fecha_hecho', 'hora_hecho', 'tipo_lugar', 'tipo_lugar_ampliado', 'tipo_lugar_otro', 'modalidad', 'modalidad_ampliado', 'modalidad_otro', 'motivo_origen_registro', 'motivo_origen_registro_otro', 'suicida_sexo', 'suicida_tr_edad', 'suicida_18_aÃ±os_o_mas', 'suicida_clase', 'suicida_clase_otro', 'suicida_identidad_genero']


,id_hecho,tipo_persona_id,federal,provincia_id,provincia_nombre,departamento_id,departamento_nombre,localidad_id,localidad_nombre,anio,...,modalidad_ampliado,modalidad_otro,motivo_origen_registro,motivo_origen_registro_otro,suicida_sexo,suicida_tr_edad,suicida_18_aÃ±os_o_mas,suicida_clase,suicida_clase_otro,suicida_identidad_genero
0,8529,Suicida idRegistro 20712,No,42,La Pampa,42002.0,Norte (General Pico),105030,General Pico,2017,...,Ahorcamiento,NaN,IntervenciÃ³n policial,NaN,Masculino,20-24,Si,Sin determinaciÃ³n,NaN,Sin determinar
1,8530,Suicida idRegistro 20713,No,42,La Pampa,42003.0,Sur (General Acha),70030,GuatrachÃ©,2017,...,Arma de fuego,NaN,IntervenciÃ³n policial,NaN,Masculino,55-59,Si,Sin determinaciÃ³n,NaN,Sin determinar
2,8569,Suicida idRegistro 20752,No,78,Santa Cruz,78021.0,GÃ¼er Aike,60,28 de Noviembre,2017,...,Ahorcamiento,NaN,Denuncia particular,NaN,Masculino,15-19,No,Sin determinaciÃ³n,NaN,Sin determinar


victimas en microdatos (2020-2024): 19323
victimas en base provincial (2020-2024): 19323.0


### **Comparación de esquemas entre fuentes**

Una vez filtradas ambas bases para el período **2020-2024**, comparamos sus columnas para identificar qué variables son comunes a ambas fuentes y cuáles son exclusivas de cada una.

In [42]:
# comparacion de columnas entre la base provincial y la de microdatos
cols_prov = set(df_prov_suicidios.columns)
cols_micro = set(df_micro_2020_2024.columns)

comunes = cols_prov.intersection(cols_micro)
solo_en_prov = cols_prov - cols_micro
solo_en_micro = cols_micro - cols_prov

print(f"columnas comunes ({len(comunes)}):", sorted(list(comunes)))
print(f"columnas exclusivas de provincial ({len(solo_en_prov)}):", sorted(list(solo_en_prov)))
print(f"columnas exclusivas de microdatos ({len(solo_en_micro)}):", sorted(list(solo_en_micro)))

columnas comunes (3): ['anio', 'provincia_id', 'provincia_nombre']
columnas exclusivas de provincial (11): ['cantidad_hechos', 'cantidad_victimas', 'cantidad_victimas_fem', 'cantidad_victimas_masc', 'cantidad_victimas_sd', 'codigo_delito_snic_id', 'codigo_delito_snic_nombre', 'tasa_hechos', 'tasa_victimas', 'tasa_victimas_fem', 'tasa_victimas_masc']
columnas exclusivas de microdatos (24): ['departamento_id', 'departamento_nombre', 'fecha_hecho', 'federal', 'hora_hecho', 'id_hecho', 'localidad_id', 'localidad_nombre', 'mes', 'modalidad', 'modalidad_ampliado', 'modalidad_otro', 'motivo_origen_registro', 'motivo_origen_registro_otro', 'suicida_18_aÃ±os_o_mas', 'suicida_clase', 'suicida_clase_otro', 'suicida_identidad_genero', 'suicida_sexo', 'suicida_tr_edad', 'tipo_lugar', 'tipo_lugar_ampliado', 'tipo_lugar_otro', 'tipo_persona_id']


# Paso 2 - Modificado por Aldi

### **Unificación de la base de datos**

Para el análisis que eligimos, conservamos dos niveles de datos: los microdatos originales filtrados, que permiten analizar edad y sexo, y una tabla agregada de 120 filas (24 jurisdicciones × 5 años) para las comparaciones territoriales.

En este procedimiento, agrupamos los microdatos de suicidios por provincia y año para obtener la cantidad de víctimas registradas y luego los unimos con la base del SNIC Provincias mediante la función **pd.merge()**, utilizando como claves las columnas "*anio*" y "*provincia_id*".

El objetivo de esta unión es verificar que las cantidades de suicidios coincidan entre ambas fuentes y detectar posibles diferencias o inconsistencias en los registros.
Utilizamos **merge()** en lugar de **concat()** porque las bases contienen distintos niveles de información: la base provincial presenta datos agregados,
mientras que los microdatos contienen registros individuales. Por lo tanto, **concat()** simplemente apilaría las filas de ambas tablas, mezclando diferentes unidades de observación y pudiendo generar duplicaciones al contabilizar los casos.

En cambio, **merge()** permite relacionar los datos mediante columnas comunes, manteniendo una estructura consistente para su comparación. Finalmente, conservamos los microdatos como base principal para analizar las cantidades de suicidios según año, provincia, región y grupo de edad, mientras que la tabla unificada se utiliza para validar la consistencia de la información.

### **Integración y validación de las bases de datos**

*(Todo esto lo podemos agregar en el pdf de la presentación)*

Para el análisis elegido, conservamos dos niveles de información: los microdatos de suicidios correspondientes al período 2020–2024, que permiten analizar variables como edad y sexo, y una tabla provincial agregada que permite realizar comparaciones territoriales. Esta última contempla las 120 combinaciones esperadas de provincias y jurisdicciones (24 jurisdicciones × 5 años).

En primer lugar, utilizamos la función groupby() para agrupar los microdatos por provincia y año, obteniendo la cantidad de víctimas registradas en cada combinación. Esta agrupación se realiza únicamente para facilitar la comparación entre ambas fuentes, sin modificar ni perder los registros individuales de la base original.

Posteriormente, relacionamos los datos agrupados con la base SNIC Provincias mediante la función pd.merge(), utilizando como claves las columnas anio y provincia_id.

El objetivo de esta operación es verificar que las cantidades de suicidios coincidan entre ambas fuentes e identificar posibles diferencias o inconsistencias en los registros.

Utilizamos merge() en lugar de concat() porque ambas bases presentan diferentes unidades de observación: la base provincial contiene información agregada, mientras que los microdatos contienen registros individuales. Por lo tanto, concat() simplemente apilaría las filas de ambas tablas, mezclando distintos niveles de información y pudiendo generar errores al contabilizar los casos.

En cambio, merge() permite relacionar los datos mediante columnas comunes y comparar sus cantidades de manera consistente, una vez que los microdatos fueron agrupados al mismo nivel que la base provincial.

Finalmente, conservamos los microdatos como fuente principal para analizar las cantidades de suicidios según año, provincia, región y grupo de edad, mientras que la tabla resultante del merge() se utiliza para validar la consistencia de la información entre ambas fuentes.

In [46]:
# Agrupamos los microdatos por provincia y año
resumen_micro = (
    df_micro_2020_2024.groupby(["anio", "provincia_id"])
    .agg(
        victimas_micro=("tipo_persona_id", "size")
    )
    .reset_index()
)

# Unificamos la información de ambas fuentes
unificado = df_prov_suicidios.merge(
    resumen_micro,
    on=["anio", "provincia_id"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

# Comprobamos la correspondencia entre ambas bases
print("Correspondencia entre las bases:")
print(unificado["_merge"].value_counts())

# Calculamos las diferencias entre las cantidades
unificado["diferencia"] = (
    unificado["cantidad_victimas"] -
    unificado["victimas_micro"]
)

print("\nDiferencias encontradas:")
print(unificado["diferencia"].value_counts(dropna=False))

# Visualizamos la tabla resultante
display(unificado.head())

Correspondencia entre las bases:
_merge
both          120
left_only       0
right_only      0
Name: count, dtype: int64

Diferencias encontradas:
diferencia
0.0    120
Name: count, dtype: int64


,provincia_id,provincia_nombre,anio,codigo_delito_snic_id,codigo_delito_snic_nombre,cantidad_hechos,cantidad_victimas,cantidad_victimas_masc,cantidad_victimas_fem,cantidad_victimas_sd,tasa_hechos,tasa_victimas,tasa_victimas_masc,tasa_victimas_fem,victimas_micro,_merge,diferencia
0,2,Ciudad AutÃ³noma de Buenos Aires,2020,31,Suicidios (consumados),206,206.0,144.0,62.0,0.0,"7,17360364586217","7,17360364586217","10,7500391928512","4,04670947040974",206,both,0.0
1,6,Buenos Aires,2020,31,Suicidios (consumados),858,858.0,688.0,166.0,4.0,"5,32063326698045","5,32063326698045","8,73216236897473","2,0128577960832",858,both,0.0
2,10,Catamarca,2020,31,Suicidios (consumados),48,48.0,39.0,9.0,0.0,"12,5954829449289","12,5954829449289","20,4989119808256","4,716116016454",48,both,0.0
3,14,CÃ³rdoba,2020,31,Suicidios (consumados),197,197.0,158.0,35.0,4.0,"5,68595571823867","5,68595571823867","9,36368986273897","1,96927037969784",197,both,0.0
4,18,Corrientes,2020,31,Suicidios (consumados),36,36.0,32.0,4.0,0.0,"3,53085179838052","3,53085179838052","6,38838591440761","0,771195835542488",36,both,0.0


In [47]:
# Guardamos los microdatos para los análisis posteriores
df_micro_2020_2024.to_csv(
    "suicidios_microdatos_2020_2024.csv",
    index=False
)

# Guardamos la tabla de validación provincial
unificado.to_csv(
    "validacion_provincias_2020_2024.csv",
    index=False
)

print("Archivos guardados correctamente.")


Archivos guardados correctamente.
